# BDW Sovereign 7B — Kaggle Backend (Ollama Edition)

> **Architecture:** Ollama serves the `bdw-7b-sft-Q4_K_M.gguf` model natively on port 11434  
> Cloudflared exposes it as `https://ai.berojgardegreewala.app` (OpenAI-compatible `/v1` API)  
> Frontend at `bdw-llm-frontend.vercel.app` routes `bdw-7b-sft` model requests here.

**Required Kaggle Secrets (add in Notebook settings → Add-ons → Secrets):**
- `HF_TOKEN` — Hugging Face token to pull the model
- `CF_TUNNEL_TOKEN` — Cloudflare tunnel token
- `BDW_API_KEY` — Optional bearer key for request auth

**Why Ollama instead of llama-cpp-python?**  
Kaggle now uses Python 3.13 by default, which breaks `llama-cpp-python`'s C++ compilation.  
Ollama ships pre-compiled binaries — zero compilation needed, instant model serving.

In [ ]:
# Cell 1: Load secrets safely (never print them)
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN        = secrets.get_secret("HF_TOKEN")
CF_TUNNEL_TOKEN = secrets.get_secret("CF_TUNNEL_TOKEN")
BDW_API_KEY     = secrets.get_secret("BDW_API_KEY")

print("✅ Secrets loaded (HF, CF, BDW)")

In [ ]:
# Cell 2: Install Ollama (pre-compiled binary — no C++ compilation needed)
import subprocess, os, time

print("📦 Installing Ollama...")
result = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True, capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("Ollama install failed")
print("✅ Ollama installed")

# Verify
ver = subprocess.run("ollama --version", shell=True, capture_output=True, text=True)
print("Version:", ver.stdout.strip())

In [ ]:
# Cell 3: Start Ollama server in background
import subprocess, time, requests

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"  # Listen on all interfaces
env["OLLAMA_MODELS"] = "/kaggle/working/ollama_models"  # Store models in /kaggle/working
os.makedirs("/kaggle/working/ollama_models", exist_ok=True)

print("🚀 Starting Ollama server...")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for Ollama to be ready
for i in range(30):
    try:
        r = requests.get("http://localhost:11434", timeout=2)
        if r.status_code == 200:
            print(f"✅ Ollama server ready (attempt {i+1})")
            break
    except:
        pass
    time.sleep(1)
else:
    print("❌ Ollama server did not start in time")
    print(ollama_proc.stderr.read(500))

In [ ]:
# Cell 4: Download model from Hugging Face and register with Ollama
import subprocess, os, shutil

MODEL_REPO   = "aamitkr26/bdw-7b-sft"
MODEL_FILE   = "bdw-7b-sft-Q4_K_M.gguf"
MODEL_PATH   = f"/kaggle/working/{MODEL_FILE}"
OLLAMA_NAME  = "bdw-7b-sft"  # This is the model name Ollama will use

if not os.path.exists(MODEL_PATH):
    print(f"📥 Downloading {MODEL_FILE} from HF...")
    subprocess.run([
        "python", "-c",
        f"""
from huggingface_hub import hf_hub_download
import os
os.environ['HF_TOKEN'] = '{HF_TOKEN}'
path = hf_hub_download(repo_id='{MODEL_REPO}', filename='{MODEL_FILE}', local_dir='/kaggle/working')
print('Downloaded to:', path)
"""
    ], check=True)
else:
    print(f"✅ Model already exists at {MODEL_PATH}")

print(f"Model size: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB")

In [ ]:
# Cell 5: Create Modelfile and register GGUF with Ollama
import subprocess

MODELFILE_CONTENT = f"""FROM {MODEL_PATH}
SYSTEM You are BerojgarDegreeWala AI, India's sovereign deep-tech hardware career and research intelligence. You are running on the BDW Kaggle inference cluster.
PARAMETER temperature 0.1
PARAMETER num_ctx 4096
PARAMETER num_predict 512
"""

with open("/kaggle/working/Modelfile", "w") as f:
    f.write(MODELFILE_CONTENT)

print("📋 Creating Ollama model 'bdw-7b-sft' from GGUF...")
result = subprocess.run(
    ["ollama", "create", "bdw-7b-sft", "-f", "/kaggle/working/Modelfile"],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Model registration failed")
print("✅ Model registered")

# Verify model is listed
models = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(models.stdout)

In [ ]:
# Cell 6: Health check — test Ollama OpenAI-compatible endpoint
import requests, json

print("🔍 Testing local Ollama /v1/chat/completions endpoint...")
try:
    r = requests.post(
        "http://localhost:11434/v1/chat/completions",
        json={
            "model": "bdw-7b-sft",
            "messages": [{"role": "user", "content": "Say 'BDW online' and nothing else."}],
            "max_tokens": 20,
            "stream": False
        },
        timeout=60
    )
    if r.status_code == 200:
        content = r.json()["choices"][0]["message"]["content"]
        print(f"✅ Model response: {content}")
    else:
        print(f"❌ Status {r.status_code}: {r.text[:500]}")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# Cell 7: Install cloudflared and start tunnel
# Tunnel routes external HTTPS → localhost:11434 (Ollama)
import subprocess, time, threading

print("📦 Installing cloudflared...")
subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
ver = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True)
print("Cloudflared:", ver.stdout.strip())

print("\n🌐 Starting Cloudflare tunnel...")
print("Tunnel routes: ai.berojgardegreewala.app → localhost:11434")

def stream_logs(proc, label):
    for line in iter(proc.stderr.readline, b''):
        decoded = line.decode('utf-8', errors='replace').strip()
        if decoded:
            print(f"[{label}] {decoded}")

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "run", "--token", CF_TUNNEL_TOKEN],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Stream tunnel logs in background thread
log_thread = threading.Thread(target=stream_logs, args=(tunnel_proc, "CF"), daemon=True)
log_thread.start()

time.sleep(5)
print("\n✅ Tunnel process started (PID:", tunnel_proc.pid, ")")
print("🌐 Public endpoint: https://ai.berojgardegreewala.app/v1/chat/completions")
print("🌐 Alternate:       https://kaggle.berojgardegreewala.app/v1/chat/completions")

In [ ]:
# Cell 8: Final public health check via Cloudflare tunnel
import requests, time

PUBLIC_URL = "https://ai.berojgardegreewala.app"

print(f"🌐 Testing public endpoint {PUBLIC_URL}...")
time.sleep(3)  # Let tunnel stabilize

headers = {}
if BDW_API_KEY:
    headers["Authorization"] = f"Bearer {BDW_API_KEY}"

for attempt in range(5):
    try:
        r = requests.get(f"{PUBLIC_URL}/v1/models", headers=headers, timeout=15)
        if r.status_code == 200:
            print(f"✅ Public endpoint live! Models: {r.json()}")
            break
        else:
            print(f"Attempt {attempt+1}: HTTP {r.status_code}")
    except Exception as e:
        print(f"Attempt {attempt+1}: {e}")
    time.sleep(5)

print("\n📋 Summary:")
print(f"  Model:           bdw-7b-sft (Q4_K_M GGUF)")
print(f"  Local endpoint:  http://localhost:11434/v1")
print(f"  Public endpoint: {PUBLIC_URL}/v1")
print(f"  Frontend:        https://bdw-llm-frontend.vercel.app")
print(f"  Vercel env var:  KAGGLE_AI_BASE_URL = {PUBLIC_URL}/v1")

In [ ]:
# Cell 9: Keep-alive loop (run this last — keeps the kernel alive)
# Kaggle notebooks auto-shutdown after ~60 minutes of inactivity.
# This cell keeps the tunnel and model server alive.
import time, requests

print("♾️  Keep-alive loop started. Sending heartbeat every 5 minutes.")
print("   Stop the kernel manually to shut down the server.")

while True:
    try:
        r = requests.get("http://localhost:11434", timeout=5)
        status = "✅ alive" if r.status_code == 200 else f"⚠️ HTTP {r.status_code}"
    except Exception as e:
        status = f"❌ {e}"
    print(f"[{time.strftime('%H:%M:%S')}] Ollama: {status}")
    time.sleep(300)  # 5 minutes